## <b> <span style='color:#2ae4f5'>|</span> Rice Variety Classification and Quality Evaluation Using Image Analysis </b> 


# <b>1 <span style='color:#2ae4f5'>|</span> Import Libraries </b> 

In [ ]:
# import requirement libraries and tools
import os
from tensorflow import keras
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style= "darkgrid", color_codes = True)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten
import warnings
warnings.filterwarnings('ignore')

# <b>2 <span style='color:#2ae4f5'>|</span> Create a dataframe with the Images and Label </b> 

In [ ]:
# Set the path to the dataset
dataset_path = '/kaggle/input/rice-image-dataset/Rice_Image_Dataset'

# Initialize empty lists for storing the images and labels
images = []
labels = []

# Loop over the subfolders in the dataset
for subfolder in os.listdir(dataset_path):
    
    subfolder_path = os.path.join(dataset_path, subfolder)
    if not os.path.isdir(subfolder_path):
        continue
  
  # Loop over the images in the subfolder
    for image_filename in os.listdir(subfolder_path):
       # Load the image and store it in the images list
        image_path = os.path.join(subfolder_path, image_filename)
        images.append(image_path)
    
        # Store the label for the image in the labels list
        labels.append(subfolder)
 
 # Create a pandas DataFrame from the images and labels
df = pd.DataFrame({'image': images, 'label': labels})

# <b>3 <span style='color:#2ae4f5'>|</span> Visualization of Dataset </b> 

In [ ]:
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Tạo figure
fig, ax = plt.subplots(figsize=(8, 6), facecolor="white")

# Vẽ countplot
sns.countplot(
    x=df.label,
    edgecolor='black',
    linewidth=1,
    ax=ax
)

# Thêm giá trị trên từng cột
for p in ax.patches:
    ax.annotate(
        f'{p.get_height()}',                    # giá trị
        (p.get_x() + p.get_width() / 2, p.get_height()),  # tọa độ
        ha='center', va='bottom',               # căn giữa
        fontsize=10, color='black', xytext=(0, 3),  # offset 3px
        textcoords='offset points'
    )

# Set labels
ax.set_xlabel("Name of Class", labelpad=40)
ax.set_ylabel("The Number Of Samples for each class", labelpad=15)

# Nền vùng vẽ
ax.set_facecolor("#f9f9f9")

# Xoay nhãn trục X
plt.xticks(rotation=0)

# Save với nền trắng
plt.savefig('class.png', facecolor='white')

# Hiển thị
plt.show()


In [ ]:
from matplotlib.gridspec import GridSpec
# Create figure and grid of subplots
fig = plt.figure(figsize=(15, 15))
gs = GridSpec(5, 4, figure=fig)

# Loop through each unique category in the DataFrame
for i, category in enumerate(df['label'].unique()):
    # Get the filepaths for the first four images in the category
    filepaths = df[df['label'] == category]['image'].values[:4]
    
    # Loop through the filepaths and add an image to each subplot
    for j, filepath in enumerate(filepaths):
        ax = fig.add_subplot(gs[i, j])
        ax.imshow(plt.imread(filepath))
        ax.axis('off')
    
    # Add a label to the bottom of the subplot grid
    ax.text(300, 100, category, fontsize=25, color='darkblue')

plt.show()

# <b>4 <span style='color:#2ae4f5'>|</span> Split Data into Train and Test </b> 
**I divided our data into two separate datasets:** the **training dataset** and the **testing dataset**. The training dataset consists of **80%** of the data, while the testing dataset contains the remaining **20%**.
To facilitate the training process, I applied the **LabelEncoder to labels**. This process allowed us to convert the rice types' labels, namely **'Arborio'**, **'Basmati'**, **'Ipsala'**, **'Jasmine'**, and **'Karacadag'**, into numerical values. By assigning integer values to the labels, we enabled the utilization of these labels as target variables during the training of our machine learning model.

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['image'], df['label'], test_size=0.2, random_state=42)

# Create a dataframe for the training data
df_train = pd.DataFrame({'image': X_train, 'label': y_train})

# Create a dataframe for the test data
df_test = pd.DataFrame({'image': X_test, 'label': y_test})

# Encode the labels
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import time
st=time.time()
# Set the image size and batch size
image_size = (75, 75) # 50,50
batch_size = 32

# Create an ImageDataGenerator object with data augmentation options for image preprocessing
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)


# Create a generator for the training data
train_generator = datagen.flow_from_dataframe(
    df_train,
    x_col='image',
    y_col='label',
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

# Create a generator for the test data
test_generator = datagen.flow_from_dataframe(
    df_test,
    x_col='image',
    y_col='label',
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)
end=time.time()
print('Time:',end-st)

# <b>5 <span style='color:#2ae4f5'>|</span> Training CNN Model </b>

In [ ]:
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
num_classes = df_train['label'].nunique()
st=time.time()
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(75, 75, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    verbose=1
)
end=time.time()
print('Time:',end-st)

# <b>6 <span style='color:#2ae4f5'>|</span> Evaluate The Model </b> 

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

# Evaluate model on test data
metrics = model.evaluate(test_generator)
print('Accuracy:', metrics[1])

# Predict trên test data
y_pred_probs = model.predict(test_generator)

# Lấy nhãn dự đoán
y_pred = np.argmax(y_pred_probs, axis=1)

# Lấy nhãn thật
y_true = test_generator.classes

# Tính F1-score (macro cho đa lớp)
f1 = f1_score(y_true, y_pred, average='macro')

print('F1-score (macro):', f1)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Compute confusion matrix (you already did this)
cm = confusion_matrix(y_true, y_pred)

# Get class labels from generator
class_labels = list(test_generator.class_indices.keys())

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.savefig('CM-CNN.png')
plt.show()


In [ ]:
from sklearn.metrics import classification_report, precision_recall_fscore_support
import pandas as pd
import numpy as np
class_labels = list(test_generator.class_indices.keys())

# Compute precision, recall, and f1-score per class
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average=None, labels=range(len(class_labels))
)

# Build results table
results_df = pd.DataFrame({
    'Class': class_labels,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1
})

# Compute macro average
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='macro'
)

# Append macro average to the table
results_df.loc[len(results_df)] = ['Macro Average', macro_precision, macro_recall, macro_f1]

print(results_df)

In [ ]:
import matplotlib.pyplot as plt

# Access the dictionary inside the History object
history_dict = history.history

# Create plot with white background (default)
plt.figure(figsize=(10,6), facecolor='white')

# Plot train accuracy in red, validation accuracy in blue
plt.plot(history_dict['accuracy'], color='red', marker='o', label='Train')
plt.plot(history_dict['val_accuracy'], color='blue', marker='h', label='Validation')

plt.title('Accuracy comparison between Validation and Train Data set', fontsize=15)
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='best')

# Save figure with white background
plt.savefig('training-CNN', facecolor='white')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Access history dictionary
history_dict = history.history

# Plot
plt.figure(figsize=(10,6), facecolor='white')
plt.plot(history_dict['loss'], color='red', marker='o', label='Train')
plt.plot(history_dict['val_loss'], color='blue', marker='h', label='Validation')

plt.title('Loss comparison between Validation and Train Data set', fontsize=15)
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='best')

plt.show()


# 7. ResNet50

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

# Number of classes in your dataset
num_classes = 5  # change this to your actual number

# Build ResNet50 from scratch (no pretrained weights)
base_model = ResNet50(
    weights=None,                # do not load ImageNet weights
    include_top=False,           # exclude the default fully connected layers
    input_shape=(75, 75, 3)
)

# You can decide whether to freeze the base model or not; usually, you train the whole thing
base_model.trainable = True

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', save_best_only=True)
]

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    callbacks=callbacks
)


In [ ]:
# Lưu toàn bộ model (kiến trúc + trọng số + optimizer)
model.save("InceptionResNetV2_TrainingRice.keras")

print("Model saved successfully.")
#from tensorflow.keras.models import load_model

#model = load_model("InceptionResNetV2_TrainingRice.keras")

#print("Model loaded successfully.")

In [ ]:
import matplotlib.pyplot as plt

# Access the dictionary inside the History object
history_dict = history.history

# Create plot with white background (default)
plt.figure(figsize=(10,6), facecolor='white')

# Plot train accuracy in red, validation accuracy in blue
plt.plot(history_dict['accuracy'], color='red', marker='o', label='Train')
plt.plot(history_dict['val_accuracy'], color='blue', marker='h', label='Validation')

plt.title('Accuracy comparison between Validation and Train Data set', fontsize=15)
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='best')

# Save figure with white background
plt.savefig('training-ResNet50', facecolor='white')
plt.show()


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

# Evaluate model on test data
metrics = model.evaluate(test_generator)
print('Accuracy:', metrics[1])

# Predict trên test data
y_pred_probs = model.predict(test_generator)

# Lấy nhãn dự đoán
y_pred = np.argmax(y_pred_probs, axis=1)

# Lấy nhãn thật
y_true = test_generator.classes

# Tính F1-score (macro cho đa lớp)
f1 = f1_score(y_true, y_pred, average='macro')

print('F1-score (macro):', f1)


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

# Evaluate model on test data
metrics = model.evaluate(test_generator)
print('Accuracy:', metrics[1])

# Predict trên test data
y_pred_probs = model.predict(test_generator)

# Lấy nhãn dự đoán
y_pred = np.argmax(y_pred_probs, axis=1)

# Lấy nhãn thật
y_true = test_generator.classes

# Tính F1-score (macro cho đa lớp)
f1 = f1_score(y_true, y_pred, average='macro')

print('F1-score (macro):', f1)


In [ ]:
from sklearn.metrics import classification_report, precision_recall_fscore_support
import pandas as pd
import numpy as np
class_labels = list(test_generator.class_indices.keys())

# Compute precision, recall, and f1-score per class
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average=None, labels=range(len(class_labels))
)

# Build results table
results_df = pd.DataFrame({
    'Class': class_labels,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1
})

# Compute macro average
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='macro'
)

# Append macro average to the table
results_df.loc[len(results_df)] = ['Macro Average', macro_precision, macro_recall, macro_f1]

print(results_df)

# 8. AlexNet

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

# Number of classes in your dataset
num_classes = 5  # đổi thành số lớp thực tế của bạn

# Xây dựng AlexNet từ đầu
model = Sequential([
    Conv2D(96, (11, 11), strides=4, activation='relu', input_shape=(75, 75, 3), padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(3, 3), strides=2),

    Conv2D(256, (5, 5), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(3, 3), strides=2),

    Conv2D(384, (3, 3), activation='relu', padding='same'),
    Conv2D(384, (3, 3), activation='relu', padding='same'),
    Conv2D(256, (3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(3, 3), strides=2),

    Flatten(),
    Dense(4096, activation='relu'),
    Dropout(0.5),
    Dense(4096, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# Compile
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True),
    # Lưu full model
    ModelCheckpoint('best_model.keras', save_best_only=True)
]

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    callbacks=callbacks
)


In [ ]:
# Lưu toàn bộ model (kiến trúc + trọng số + optimizer)
model.save("AlexNetRice.keras")

print("Model saved successfully.")
#from tensorflow.keras.models import load_model

#model = load_model("InceptionResNetV2_TrainingRice.keras")

#print("Model loaded successfully.")

In [ ]:
import matplotlib.pyplot as plt

# Access the dictionary inside the History object
history_dict = history.history

# Create plot with white background (default)
plt.figure(figsize=(10,6), facecolor='white')

# Plot train accuracy in red, validation accuracy in blue
plt.plot(history_dict['accuracy'], color='red', marker='o', label='Train')
plt.plot(history_dict['val_accuracy'], color='blue', marker='h', label='Validation')

plt.title('Accuracy comparison between Validation and Train Data set', fontsize=15)
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='best')

# Save figure with white background
plt.savefig('training-ALEXNET', facecolor='white')
plt.show()


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

# Evaluate model on test data
metrics = model.evaluate(test_generator)
print('Accuracy:', metrics[1])

# Predict trên test data
y_pred_probs = model.predict(test_generator)

# Lấy nhãn dự đoán
y_pred = np.argmax(y_pred_probs, axis=1)

# Lấy nhãn thật
y_true = test_generator.classes

# Tính F1-score (macro cho đa lớp)
f1 = f1_score(y_true, y_pred, average='macro')

print('F1-score (macro):', f1)


In [ ]:
from sklearn.metrics import classification_report, precision_recall_fscore_support
import pandas as pd
import numpy as np
class_labels = list(test_generator.class_indices.keys())

# Compute precision, recall, and f1-score per class
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average=None, labels=range(len(class_labels))
)

# Build results table
results_df = pd.DataFrame({
    'Class': class_labels,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1
})

# Compute macro average
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='macro'
)

# Append macro average to the table
results_df.loc[len(results_df)] = ['Macro Average', macro_precision, macro_recall, macro_f1]

print(results_df)

# 9. Inception V3

In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

# Số lớp trong dataset của bạn
num_classes = 5   # thay bằng số lớp thực tế

base_model = InceptionV3(
    weights="/kaggle/input/modelv3/keras/default/1/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5",
    include_top=False,
    input_shape=(75, 75, 3)
)


# Freeze base layers (nếu muốn fine-tune sau thì bỏ dòng này)
base_model.trainable = False

# Build model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(num_classes, activation="softmax")
])

# Compile
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint  

# Số lớp trong dataset
num_classes = 5  

base_model.trainable = True  # cho phép fine-tune

# Xây dựng mô hình
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# Compile
model.compile(
    optimizer=Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', save_best_only=True)
]

# Train
history = model.fit(
    train_generator,
    epochs=15,
    validation_data=test_generator,
    callbacks=callbacks
)

In [ ]:
import matplotlib.pyplot as plt

# Access the dictionary inside the History object
history_dict = history.history

# Create plot with white background (default)
plt.figure(figsize=(10,6), facecolor='white')

# Plot train accuracy in red, validation accuracy in blue
plt.plot(history_dict['accuracy'], color='red', marker='o', label='Train')
plt.plot(history_dict['val_accuracy'], color='blue', marker='h', label='Validation')

plt.title('Accuracy comparison between Validation and Train Data set', fontsize=15)
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='best')

# Save figure with white background
plt.savefig('training-InceptionV3', facecolor='white')
plt.show()


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

# Evaluate model on test data
metrics = model.evaluate(test_generator)
print('Accuracy:', metrics[1])

# Predict trên test data
y_pred_probs = model.predict(test_generator)

# Lấy nhãn dự đoán
y_pred = np.argmax(y_pred_probs, axis=1)

# Lấy nhãn thật
y_true = test_generator.classes

# Tính F1-score (macro cho đa lớp)
f1 = f1_score(y_true, y_pred, average='macro')

print('F1-score (macro):', f1)


In [ ]:
from sklearn.metrics import classification_report, precision_recall_fscore_support
import pandas as pd
import numpy as np
class_labels = list(test_generator.class_indices.keys())

# Compute precision, recall, and f1-score per class
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average=None, labels=range(len(class_labels))
)

# Build results table
results_df = pd.DataFrame({
    'Class': class_labels,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1
})

# Compute macro average
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='macro'
)

# Append macro average to the table
results_df.loc[len(results_df)] = ['Macro Average', macro_precision, macro_recall, macro_f1]

print(results_df)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True, fmt="d", cmap="Blues",
    xticklabels=class_labels,
    yticklabels=class_labels
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# 10. Inception resnet V2 

In [ ]:
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, LeakyReLU
from tensorflow.keras.optimizers import Adam

# Number of classes in your dataset
num_classes = 5  # change this to your actual number

# Build InceptionResNetV2 from scratch (no pretrained weights)
base_model = InceptionResNetV2(
    weights=None,                # do not load ImageNet weights
    include_top=False,           # exclude the default fully connected layers
    input_shape=(75, 75, 3)      # ⚠️ Minimum size is 75x75 for InceptionResNetV2
)

# You can decide whether to freeze the base model or not; usually, you train the whole thing
base_model.trainable = True

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128),
    LeakyReLU(alpha=0.1),   # 👈 thay relu bằng LeakyReLU
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint  # ✅ Fix here

# Number of classes in your dataset
num_classes = 5

base_model = InceptionResNetV2(
    weights=None,                
    include_top=False,           
    input_shape=(75, 75, 3)      
)

base_model.trainable = True

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128),
    LeakyReLU(alpha=0.1),   # 👈 thay relu bằng LeakyReLU
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Define callbacks
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', save_best_only=True)
]

# Train
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    callbacks=callbacks
)


In [ ]:
# Save full model
model.save("InceptionResNetV2_TrainingRice.h5")

# OR just save weights
model.save_weights("InceptionResNetV2_TrainingRice.weights.h5")


print("Model saved successfully.")

In [ ]:
import matplotlib.pyplot as plt

# Access the dictionary inside the History object
history_dict = history.history

# Create plot with white background (default)
plt.figure(figsize=(10,6), facecolor='white')

# Plot train accuracy in red, validation accuracy in blue
plt.plot(history_dict['accuracy'], color='red', marker='o', label='Train')
plt.plot(history_dict['val_accuracy'], color='blue', marker='h', label='Validation')

plt.title('Accuracy comparison between Validation and Train Data set', fontsize=15)
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='best')

# Save figure with white background
plt.savefig('training-InceptionResNetv2', facecolor='white')
plt.show()


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

# Evaluate model on test data
metrics = model.evaluate(test_generator)
print('Accuracy:', metrics[1])

# Predict trên test data
y_pred_probs = model.predict(test_generator)

# Lấy nhãn dự đoán
y_pred = np.argmax(y_pred_probs, axis=1)

# Lấy nhãn thật
y_true = test_generator.classes

# Tính F1-score (macro cho đa lớp)
f1 = f1_score(y_true, y_pred, average='macro')

print('F1-score (macro):', f1)


In [ ]:
from sklearn.metrics import classification_report, precision_recall_fscore_support
import pandas as pd
import numpy as np
class_labels = list(test_generator.class_indices.keys())

# Compute precision, recall, and f1-score per class
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average=None, labels=range(len(class_labels))
)

# Build results table
results_df = pd.DataFrame({
    'Class': class_labels,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1
})

# Compute macro average
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='macro'
)

# Append macro average to the table
results_df.loc[len(results_df)] = ['Macro Average', macro_precision, macro_recall, macro_f1]

print(results_df)